---

# NLP EDA Template

*A dataset-agnostic exploratory analysis notebook for any text dataset (reviews, comments, lyrics, tweets, documents), with or without labels, in English or Indonesian. Every chart, including the word clouds, is exported to `eda-output/`.*

---

## Team Name

- Name 1 (Role)
- Name 2
- Name 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Text Toolkit**](#3)
4. [**Corpus Overview**](#4)
5. [**Text Quality and Noise**](#5)
6. [**Length Analysis**](#6)
7. [**Vocabulary Statistics**](#7)
8. [**Word Clouds**](#8)
9. [**N-gram Analysis**](#9)
10. [**Distinctive Words per Label**](#10)
11. [**Sentiment Analysis**](#11)
12. [**Topic Modeling**](#12)
13. [**Semantic Map**](#13)
14. [**Duplicates and Label Noise**](#14)
15. [**Segment and Temporal Analysis**](#15)
16. [**EDA Summary and Modelling Recommendations**](#16)


---

# Introduction <a name="1"></a>

---

## Overview

*Text data hides its problems: encoding noise, duplicates, mixed languages, label noise, and extreme length variation all silently hurt NLP models. This notebook profiles a text column end to end so that cleaning, tokenisation, sequence length, and model choices are all grounded in evidence.*

## Aim

*The notebook measures corpus size, quality, and noise, length distributions, vocabulary behaviour (Zipf, Heaps, coverage), frequent and distinctive words and phrases per label, lexicon sentiment, latent topics, and a 2-D semantic map. It closes with an automatic summary and concrete modelling recommendations (max sequence length, vocabulary size, cleaning steps, class weighting).*

## How to Point the Notebook at a Dataset

*All changes happen in the `Settings` class (section 2).*

| Setting | Purpose | Example |
|---|---|---|
| `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` | Data file per environment (auto-detected) | `'/kaggle/input/comp'` (folder) or a file path |
| `TEXT_COL` | Text column, or `'auto'` to pick the column with the longest texts | `'review'` |
| `LABEL_COL` | Class label, numeric target (binned into quantiles), `None`, or `'auto'` (from sample_submission / test) | `'sentiment'` |
| `GROUP_COL`, `TIME_COL` | Optional segment and time columns for section 15 | `'app'`, `'date'` |
| `LANGUAGE` | Stopword set: `'en'`, `'id'` (Indonesian, including slang), or `'both'` | `'id'` |
| `EXTRA_STOPWORDS`, `REMOVE_PATTERNS` | Dataset-specific stopwords and regex patterns to strip | `['app']`, `[r'\[.*?\]']` |

*Every column setting defaults to `'auto'`, so the notebook runs on any text dataset as is. `REMOVE_PATTERNS` is the place for dataset-specific noise, for example `[r'\[[^\]]*\]']` to strip `[Chorus]` style headers or `[r'@\w+']` to strip user mentions.*

## Dataset

*Any text dataset works: one document per row, a text column, and optionally a label (a class, a numeric score, ordered levels, or several labels per document). Put the files in `data/` next to this notebook, or point `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` at the dataset folder.*

```
data/
├── train.csv              (text + label)
├── test.csv               (text only, optional)
└── sample_submission.csv  (id + target columns, optional)
```

*With `'auto'` the text column is the string column with the longest documents and the label comes from `sample_submission.csv` (or is the only train column missing from the test file). Set `TEXT_COL` and `LABEL_COL` explicitly whenever the guess is wrong.*

## Approach: Layered Text Profiling

*The analysis moves from surface properties to meaning. Cheap checks run on the full corpus, while expensive ones (topic modeling, t-SNE, sentiment) run on a reproducible sample.*

```
Raw text
    |
    +-- Overview     : size, empties, label balance, examples
    +-- Quality      : URLs, HTML, emoji, numbers, casing, repeats, script / language mix
    +-- Length       : chars, words, sentences, by label
    +-- Vocabulary   : Zipf, Heaps, hapax share, coverage curve
    +-- Words        : word clouds, n-grams, log-odds distinctive words per label
    +-- Meaning      : VADER sentiment, NMF topics, TF-IDF + SVD + t-SNE map
    +-- Integrity    : exact and near duplicates, conflicting labels
    +-- Segments     : group and time breakdowns
         |
    EDA summary + recommendations -> eda-output/ (PNG + CSV + MD)
```

---

# Initialization <a name="2"></a>

---

## Environment Setup

The notebook runs on CPU only. The `nvidia-smi` call is kept for consistency with the other templates and fails harmlessly on CPU machines.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scipy scikit-learn wordcloud vaderSentiment pyarrow openpyxl

## Import Libraries

`wordcloud` and `vaderSentiment` are imported defensively, so a missing package only skips its own section instead of stopping the notebook.

In [ ]:
import os
import re
import json
import random
import warnings
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex, to_rgb
import seaborn as sns
from scipy import stats
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from IPython.display import Markdown, display

AVAILABLE = {}
try:
    from wordcloud import WordCloud
    AVAILABLE['wordcloud'] = True
except Exception:
    AVAILABLE['wordcloud'] = False
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    AVAILABLE['vader'] = True
except Exception:
    AVAILABLE['vader'] = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)
print('Optional packages:', AVAILABLE)

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

All paths, column roles, language options, and analysis knobs are centralised here. Switching between Kaggle, Colab, and local execution only requires editing the path lines in block 1.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a file or a dataset folder (train / test / sample_submission are found by name)
    KAGGLE_PATH = '/kaggle/input/<dataset-slug>'
    COLAB_PATH  = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH  = 'data'
    READ_KWARGS = {}

    # 2) Columns: LABEL_COL may be a class label, a numeric target, or None
    TEXT_COL  = 'auto'
    LABEL_COL = 'auto'
    GROUP_COL = None
    TIME_COL  = None
    ID_COL    = None

    # 3) Language and cleaning
    LANGUAGE        = 'both'
    EXTRA_STOPWORDS = []
    REMOVE_PATTERNS = []
    MIN_TOKEN_LEN   = 2

    # 4) Analysis knobs
    MAX_DOCS         = 30_000
    MAX_LABELS       = 8
    LABEL_BINS       = 4
    CLASS_MAX        = 30
    TOP_N            = 20
    WORDCLOUD_WORDS  = 150
    TOPIC_K          = 8
    SENTIMENT_SAMPLE = 5_000
    TSNE_SAMPLE      = 3_000
    NEAR_DUP_SAMPLE  = 5_000
    NEAR_DUP_SIM     = 0.90
    TIME_RANGE       = (1950, 2025)

    # 5) Outputs
    DATA_PATH  = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    OUTPUT_DIR = Path('/kaggle/working/eda-output') if _ON_KAGGLE else Path('eda-output')
    FIG_DIR    = OUTPUT_DIR / 'figures'
    TABLE_DIR  = OUTPUT_DIR / 'tables'

CFG = Settings()
for d in (CFG.OUTPUT_DIR, CFG.FIG_DIR, CFG.TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')

## Load Dataset

The path is resolved (with a Kaggle fallback to the largest tabular file), the text column is chosen, missing texts are kept as empty strings so they can be counted, and the analysis set is capped at `MAX_DOCS` rows with a fixed seed.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.jsonl', '.zip', '.gz')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.jsonl' in suffixes:
        return pd.read_json(path, lines=True, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

def file_role(f):
    """Classify a data file by its name: 'train', 'test', 'sample' (submission template), or None."""
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')

def list_tables(root, max_depth=6):
    # os.walk + extension check by name: no per-file disk call, which is slow on the /kaggle/input network mount
    root, out = Path(root), []
    for d, dirs, names in os.walk(root):
        depth = len(Path(d).relative_to(root).parts)
        dirs[:] = [x for x in dirs if x not in SKIP_DIRS and not x.startswith('.')] if depth < max_depth else []
        out += [Path(d) / n for n in names if Path(n).suffix.lower() in TABULAR_EXT]
    return out

def discover_files(data_path, test_path):
    """Resolve the train, test, and sample-submission files from a file, a dataset folder, or the input root."""
    p = Path(data_path) if data_path else None
    explicit_test = Path(test_path) if test_path and Path(test_path).exists() else None
    if test_path and explicit_test is None:
        print(f'[warn] test path {test_path} not found; searching the dataset folder instead')
    if p is not None and p.is_file():
        train, files = p, [f for f in p.parent.iterdir() if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    else:
        if p is not None and p.is_dir():
            roots = [p]
        elif CFG._ON_KAGGLE:
            roots = [Path('/kaggle/input')]
        elif CFG._ON_COLAB:
            roots = [Path('/content')]
        else:
            # local: data/, input/, dataset/ next to the notebook first, then the notebook folder itself
            roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
        if p is not None and not p.is_dir():
            print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
        train, files = None, []
        for r in (r for r in roots if r.exists()):
            files = list_tables(r)
            named = [f for f in files if file_role(f) == 'train']
            others = [f for f in files if file_role(f) is None]
            train = max(named or others, key=lambda f: f.stat().st_size, default=None)
            if train is not None:
                break
    if train is None:
        return None, explicit_test, None

    def pick(role):
        near = [f for f in files if file_role(f) == role and f.parent == train.parent]
        cands = near or [f for f in files if file_role(f) == role]
        exact = [f for f in cands if f.name.lower().split('.')[0] == role]
        return min(exact or cands, key=lambda f: len(f.name), default=None)

    test = explicit_test or pick('test')
    sample = pick('sample')
    for role, f in (('train', train), ('test', test), ('sample submission', sample)):
        print(f'{role:<18}: {f if f else "(none found)"}')
    return train, test, sample

DATA_PATH, TEST_PATH, SAMPLE_PATH = discover_files(CFG.DATA_PATH, None)
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found and no train file was discovered. Edit the path lines in Settings.'
raw = load_table(DATA_PATH, **CFG.READ_KWARGS)

if CFG.TEXT_COL == 'auto' or CFG.TEXT_COL not in raw.columns:
    obj = [c for c in raw.columns if not pd.api.types.is_numeric_dtype(raw[c])]
    TEXT_COL = max(obj, key=lambda c: raw[c].dropna().astype(str).str.len().mean())
    print(f'[info] TEXT_COL auto-selected: {TEXT_COL}')
else:
    TEXT_COL = CFG.TEXT_COL
if CFG.LABEL_COL == 'auto':
    raw_test = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
    sample = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
    only_train = [c for c in raw.columns if raw_test is not None and c not in raw_test.columns and c != TEXT_COL]
    if sample is not None and sample.shape[1] >= 2 and sample.columns[1] in raw.columns:
        LABEL_COL = sample.columns[1]
    else:
        LABEL_COL = only_train[0] if len(only_train) == 1 else None
    print(f'[info] LABEL_COL auto-selected: {LABEL_COL}' if LABEL_COL else
          '[info] no label found (no sample_submission or test file): unlabeled EDA. Set LABEL_COL in Settings.')
else:
    LABEL_COL = CFG.LABEL_COL if CFG.LABEL_COL in raw.columns else None
if CFG.LABEL_COL not in (None, 'auto') and LABEL_COL is None:
    print(f'[warn] LABEL_COL={CFG.LABEL_COL!r} not found, running unlabeled EDA')

df = raw.copy()
df['text'] = df[TEXT_COL].fillna('').astype(str)
N_TOTAL = len(df)
if len(df) > CFG.MAX_DOCS:
    df = df.sample(CFG.MAX_DOCS, random_state=CFG.SEED)
df = df.reset_index(drop=True)
print(f'Loaded {N_TOTAL:,} rows from {DATA_PATH}; analysing {len(df):,} rows')
print(f'Text column: {TEXT_COL!r} | label column: {LABEL_COL!r}')

---

# Text Toolkit <a name="3"></a>

---

The plotting identity matches the other templates. Each label gets a fixed colour from the palette, which is reused in every chart and in the per-label word clouds, so a label is recognisable across the whole report.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])
CMAP_HEAT = LinearSegmentedColormap.from_list('heat', ['#FFF8F0', '#F4A261', '#E76F51', '#9D0208'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = [0]
SAVED_FIGS = []

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name):
    FIG_COUNTER[0] += 1
    path = CFG.FIG_DIR / f'{FIG_COUNTER[0]:02d}_{slug(name)}.png'
    fig.savefig(path, facecolor='white')
    SAVED_FIGS.append(path.name)
    plt.show()
    plt.close(fig)

def save_table(table, name, index=True):
    table.to_csv(CFG.TABLE_DIR / f'{slug(name)}.csv', index=index)

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def make_grid(n, ncols=4, w=4.2, h=3.2):
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False)
    axes = axes.ravel()
    for ax in axes[n:]:
        ax.set_visible(False)
    return fig, axes[:n]

def bar_labels(ax, fmt='{:,.0f}', size=8):
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues], padding=3, fontsize=size, color='#4A5568')

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def note(msg):
    display(Markdown(f'> **Note:** {msg}'))

## Stopwords and Tokenisation

Stopwords combine scikit-learn's English list, a built-in Indonesian list that includes common informal and slang forms (`gak`, `aja`, `yg`), apostrophe-free contractions, and `EXTRA_STOPWORDS`. Tokens are Unicode letter sequences, so accented and non-Latin words survive, while digits and punctuation are dropped. Two token streams are kept: all tokens (for vocabulary statistics) and content tokens without stopwords (for word clouds, n-grams, and topics).

In [ ]:
STOP_ID = set('''yang dan di ke dari ini itu dengan untuk tidak ada saya aku kamu kau dia mereka kita kami
anda akan juga sudah telah belum bisa dapat harus atau tapi tetapi namun karena jadi jika kalau saat ketika
seperti pada dalam oleh agar supaya sangat lebih paling sama lagi masih hanya saja pun bagi tentang antara
setelah sebelum sejak hingga sampai mau ingin perlu pernah sedang apa siapa mana bagaimana kenapa mengapa
begitu sini situ sana nya lah kah pula para sebuah seorang suatu semua setiap banyak sedikit hal adalah ialah
yaitu yakni merupakan bahwa maka serta tanpa tak bukan jangan dong deh sih kok nih tuh loh lho kan ya yah
gak ga nggak ngga enggak engga gk tdk aja doang udah udh sdh blm bgt banget yg dgn utk krn klo kalo trs terus
gw gue lu lo elo aku ku mu nya min kak gan sis bro'''.split())
CONTRACTIONS = set('dont im youre cant ive ill thats wont didnt doesnt isnt arent wasnt werent couldnt wouldnt '
                   'shouldnt hes shes theyre weve youve youll itll lets aint id youd hed theyd whats theres'.split())

STOPWORDS = set(CONTRACTIONS) | {w.lower() for w in CFG.EXTRA_STOPWORDS}
if CFG.LANGUAGE in ('en', 'both'):
    STOPWORDS |= set(ENGLISH_STOP_WORDS)
if CFG.LANGUAGE in ('id', 'both'):
    STOPWORDS |= STOP_ID

URL_RE      = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE    = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.]+\b')
MENTION_RE  = re.compile(r'(?<!\w)@\w+')
HASHTAG_RE  = re.compile(r'(?<!\w)#\w+')
HTML_RE     = re.compile(r'<[^>]+>|&[a-z]+;|&#\d+;')
NUMBER_RE   = re.compile(r'\b\d+(?:[.,]\d+)?\b')
EMOJI_RE    = re.compile('[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]')
REPEAT_RE   = re.compile(r'([^\W\d_])\1{2,}')
TOKEN_RE    = re.compile(r"[^\W\d_]+(?:'[^\W\d_]+)?")
SENT_RE     = re.compile(r'[.!?]+|\n+')
REMOVE_RES  = [re.compile(p) for p in CFG.REMOVE_PATTERNS]

def clean(text):
    for rx in REMOVE_RES:
        text = rx.sub(' ', text)
    text = URL_RE.sub(' ', text)
    text = HTML_RE.sub(' ', text)
    text = unicodedata.normalize('NFKC', text).lower()
    return REPEAT_RE.sub(r'\1\1', text)

def tokens_all(text):
    return [t.replace("'", '') for t in TOKEN_RE.findall(clean(text))]

def content(tokens):
    return [t for t in tokens if len(t) >= CFG.MIN_TOKEN_LEN and t not in STOPWORDS]

df['clean'] = [clean(t) for t in df['text']]
TOK_ALL = [[t.replace("'", '') for t in TOKEN_RE.findall(c)] for c in df['clean']]
TOK = [content(t) for t in TOK_ALL]
df['n_tokens'] = [len(t) for t in TOK_ALL]
df['n_content'] = [len(t) for t in TOK]
print(f'{len(STOPWORDS):,} stopwords | {sum(df.n_tokens):,} tokens | {sum(df.n_content):,} content tokens')
print('Example:', TOK[0][:25])

## Label Preparation

A categorical label is used as is. A numeric target with many distinct values is binned into `LABEL_BINS` quantile groups (Q1 is the lowest), so every per-label analysis also works for regression targets. Per-label charts show the `MAX_LABELS` most frequent labels and assign each one a fixed colour.

In [ ]:
HAS_LABEL = LABEL_COL is not None
LABEL_KIND = None
if HAS_LABEL:
    lab = df[LABEL_COL]
    if pd.api.types.is_numeric_dtype(lab) and lab.nunique() > CFG.CLASS_MAX:
        names = [f'Q{i + 1}' for i in range(CFG.LABEL_BINS)]
        names[0], names[-1] = names[0] + ' (low)', names[-1] + ' (high)'
        df['label'] = pd.qcut(lab.rank(method='first'), CFG.LABEL_BINS, labels=names).astype(str)
        LABEL_KIND = 'binned numeric'
        ORDER = names
    else:
        df['label'] = lab.astype(str).where(lab.notna(), '(missing)')
        LABEL_KIND = 'categorical'
        ORDER = df['label'].value_counts().index.tolist()
    TOP_LABELS = ORDER[:CFG.MAX_LABELS] if LABEL_KIND == 'categorical' else ORDER
    LABEL_COLOR = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(TOP_LABELS)}
    df['label_top'] = df['label'].where(df['label'].isin(TOP_LABELS), '(other)')
    print(f'Label kind: {LABEL_KIND} | {df.label.nunique()} labels | plotted: {TOP_LABELS}')
else:
    TOP_LABELS, LABEL_COLOR = [], {}
    note('No label column, so per-label analyses are skipped.')

---

# Corpus Overview <a name="4"></a>

---

The overview establishes the basic shape of the corpus: how many documents exist, how many are empty or duplicated, and how labels are distributed. Label imbalance decides the metric (macro F1 for skewed classes) and whether class weights are needed.

In [ ]:
norm = df['clean'].str.split().str.join(' ')
overview = pd.Series({
    'documents (analysed)': len(df),
    'documents (file total)': N_TOTAL,
    'empty or whitespace': int((df['text'].str.strip() == '').sum()),
    'no content tokens': int((df['n_content'] == 0).sum()),
    'exact duplicate texts': int(df['text'].duplicated().sum()),
    'duplicates after normalisation': int(norm[norm != ''].duplicated().sum()),
    'labels': df['label'].nunique() if HAS_LABEL else 0,
}, name='value').to_frame()
overview['share_%'] = overview['value'] / len(df) * 100
overview.loc[['documents (analysed)', 'documents (file total)', 'labels'], 'share_%'] = np.nan
save_table(overview, 'corpus_overview')
display(overview)

## Label Distribution

The chart shows document counts per label with their share. The imbalance ratio (largest label over smallest) is reported in the subtitle.

In [ ]:
if HAS_LABEL:
    vc = df['label'].value_counts()
    if LABEL_KIND == 'binned numeric':
        vc = vc.reindex(ORDER)
    show = vc.head(30)[::-1]
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.3 * len(show) + 1.6)))
    ax.barh([short(i, 26) for i in show.index], show.values, color=[LABEL_COLOR.get(i, MUTED) for i in show.index])
    for i, v in enumerate(show.values):
        ax.text(v, i, f'  {v:,} ({v / len(df):.1%})', va='center', fontsize=8, color='#4A5568')
    ax.set_xlabel('documents')
    extra = f', top 30 of {len(vc)} shown' if len(vc) > 30 else ''
    suptitle(fig, f'Label distribution: {LABEL_COL}', f'imbalance ratio {vc.max() / vc.min():.1f}x{extra}; coloured = labels used in per-label charts')
    save_fig(fig, 'label_distribution')
    save_table(vc.rename('count').to_frame(), 'label_distribution')

## Example Documents

Reading raw examples is irreplaceable: it reveals formatting artefacts, boilerplate, and language that no statistic flags. Two random documents per plotted label are shown, truncated to 300 characters.

In [ ]:
groups = TOP_LABELS if HAS_LABEL else [None]
rows = []
for g in groups:
    pool = df[df['label'] == g] if g is not None else df
    for _, r in pool.sample(min(2, len(pool)), random_state=CFG.SEED).iterrows():
        rows.append({'label': g, 'words': r['n_tokens'], 'text': short(r['text'].replace('\n', ' / '), 300)})
display(pd.DataFrame(rows))

#### Insights

> *Fill in after running.* How large and how clean is the corpus? How imbalanced are the labels, and what does that imply for the metric and class weighting? What artefacts are visible in the raw examples (headers, signatures, templates, mixed languages)?

---

# Text Quality and Noise <a name="5"></a>

---

Noise patterns determine the cleaning pipeline. The prevalence of URLs, mentions, hashtags, HTML, numbers, emoji, elongated words, and shouting is measured as the share of documents containing each pattern. The script and language mix is estimated from non-ASCII letters and from the share of English versus Indonesian stopwords in each document.

In [ ]:
patterns = {'url': URL_RE, 'email': EMAIL_RE, 'mention (@user)': MENTION_RE, 'hashtag': HASHTAG_RE,
            'html / entity': HTML_RE, 'number': NUMBER_RE, 'emoji': EMOJI_RE, 'elongated word (sooo)': REPEAT_RE}
for rx in REMOVE_RES:
    patterns[f'custom: {rx.pattern}'] = rx
rows = []
for name, rx in patterns.items():
    counts = df['text'].map(lambda t: len(rx.findall(t)))
    rows.append({'pattern': name, 'docs_%': (counts > 0).mean() * 100, 'mean_per_doc': counts.mean()})

letters = df['text'].str.count(r'[^\W\d_]').clip(lower=1)
upper = df['text'].str.count(r'[A-Z]')
non_ascii = df['text'].str.count(r'[^\x00-\x7f]')
rows.append({'pattern': 'shouting (>50% caps)', 'docs_%': ((upper / letters) > 0.5).mean() * 100, 'mean_per_doc': np.nan})
rows.append({'pattern': 'non-ASCII heavy (>20%)', 'docs_%': ((non_ascii / letters) > 0.2).mean() * 100, 'mean_per_doc': np.nan})
rows.append({'pattern': 'excess punctuation (!!! ???)', 'docs_%': df['text'].str.contains(r'[!?]{2,}').mean() * 100, 'mean_per_doc': np.nan})
noise = pd.DataFrame(rows).set_index('pattern').sort_values('docs_%', ascending=False)
save_table(noise, 'noise_patterns')
display(noise.round(3))

en_sw, id_sw = set(ENGLISH_STOP_WORDS) | CONTRACTIONS, STOP_ID
def guess_language(tokens):
    if len(tokens) < 3:
        return 'too short'
    en = sum(t in en_sw for t in tokens) / len(tokens)
    idn = sum(t in id_sw for t in tokens) / len(tokens)
    if max(en, idn) < 0.08:
        return 'other / unknown'
    if en > 1.5 * idn:
        return 'English'
    if idn > 1.5 * en:
        return 'Indonesian'
    return 'mixed'
df['lang_guess'] = [guess_language(t) for t in TOK_ALL]
lang = df['lang_guess'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, max(4.5, 0.36 * len(noise) + 1.8)), gridspec_kw={'width_ratios': [1.6, 1]})
n = noise['docs_%'][::-1]
axes[0].barh(n.index, n.values, color=[ACCENT if v > 5 else PALETTE[3] if v > 1 else PALETTE[2] for v in n.values])
bar_labels(axes[0], '{:.1f}%')
axes[0].set_xlabel('% of documents')
axes[0].set_title('Noise pattern prevalence (green < 1% < yellow < 5% < orange)', loc='left')
lp = (lang / lang.sum() * 100).sort_values()
axes[1].barh(lp.index, lp.values, color=[PALETTE[i % 10] for i in range(len(lp))][::-1])
bar_labels(axes[1], '{:.1f}%')
axes[1].set_xlabel('% of documents')
axes[1].set_title('Language guess (stopword heuristic)', loc='left')
suptitle(fig, 'Text quality and noise', 'decide which patterns to strip, normalise, or keep as features')
save_fig(fig, 'noise_and_language')

#### Insights

> *Fill in after running.* Which patterns are frequent enough to need handling (above 1% of documents)? Emoji and elongations often carry sentiment and are better converted than deleted. Is the corpus monolingual, and does the language mix call for a multilingual model?

---

# Length Analysis <a name="6"></a>

---

Length drives the maximum sequence length of transformer models, the cost of training, and sometimes the label itself (complaints are often longer than praise). Characters, words, sentences, average word length, and lexical diversity (type-token ratio) are profiled.

In [ ]:
df['n_chars'] = df['text'].str.len()
df['n_sentences'] = [max(1, len([s for s in SENT_RE.split(t) if s.strip()])) for t in df['text']]
df['avg_word_len'] = [np.mean([len(w) for w in t]) if t else 0 for t in TOK_ALL]
df['ttr'] = [len(set(t)) / len(t) if t else 0 for t in TOK_ALL]
len_cols = ['n_chars', 'n_tokens', 'n_content', 'n_sentences', 'avg_word_len', 'ttr']
len_stats = df[len_cols].describe(percentiles=[0.05, 0.5, 0.9, 0.95, 0.99]).T
save_table(len_stats, 'length_stats')
display(len_stats.round(2))

titles = {'n_chars': 'Characters', 'n_tokens': 'Words', 'n_content': 'Content words (no stopwords)',
          'n_sentences': 'Sentences / lines', 'avg_word_len': 'Average word length', 'ttr': 'Type-token ratio'}
fig, axes = make_grid(len(len_cols), ncols=3, w=5.4, h=3.6)
for i, (ax, c) in enumerate(zip(axes, len_cols)):
    x = df[c].clip(upper=df[c].quantile(0.99))
    sns.histplot(x, bins=50, color=PALETTE[i], edgecolor='white', alpha=0.85, ax=ax)
    for q, ls in ((0.5, '--'), (0.95, ':')):
        ax.axvline(df[c].quantile(q), color=DARK, ls=ls, lw=1.2)
    ax.set_title(f'{titles[c]}  (median {df[c].median():,.1f}, p95 {df[c].quantile(0.95):,.1f})', loc='left', fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
suptitle(fig, 'Document length profile', 'clipped at p99; dashed = median, dotted = p95')
save_fig(fig, 'length_profile')

## Length by Label

Box plots compare word counts across labels, and a Kruskal-Wallis test with epsilon-squared effect size checks whether length alone separates the labels. A large effect means length is a useful feature and a possible shortcut that a model may exploit.

In [ ]:
if HAS_LABEL:
    d = df[df['label'].isin(TOP_LABELS)]
    groups = [g['n_tokens'].values for _, g in d.groupby('label')]
    H, p = stats.kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
    eps2 = H / (len(d) - 1) if len(groups) > 1 else np.nan
    order = d.groupby('label')['n_tokens'].median().sort_values().index.tolist() if LABEL_KIND == 'categorical' else TOP_LABELS
    fig, axes = plt.subplots(1, 2, figsize=(17, max(4.5, 0.42 * len(order) + 1.8)))
    sns.boxplot(data=d, y='label', x='n_tokens', order=order, palette=LABEL_COLOR, ax=axes[0], showfliers=False, width=0.6)
    axes[0].set_xlabel('words')
    axes[0].set_ylabel('')
    axes[0].set_title(f'Words per document (Kruskal p = {p:.2g}, eps2 = {eps2:.3f})', loc='left')
    sns.boxplot(data=d, y='label', x='ttr', order=order, palette=LABEL_COLOR, ax=axes[1], showfliers=False, width=0.6)
    axes[1].set_ylabel('')
    axes[1].set_xlabel('type-token ratio')
    axes[1].set_title('Lexical diversity (higher = less repetitive)', loc='left')
    suptitle(fig, f'Length and diversity by {LABEL_COL}', 'outliers hidden; labels sorted by median length')
    save_fig(fig, 'length_by_label')
    if LABEL_KIND == 'binned numeric':
        rho, pr = stats.spearmanr(df['n_tokens'], df[LABEL_COL], nan_policy='omit')
        print(f'Spearman rho between word count and {LABEL_COL}: {rho:+.3f} (p = {pr:.2g})')

#### Insights

> *Fill in after running.* What are the median and p95 lengths? These set the truncation length for transformers. Does length or repetitiveness differ by label enough to be a feature?

---

# Vocabulary Statistics <a name="7"></a>

---

Natural language vocabularies follow Zipf's law (frequency is inversely proportional to rank) and Heaps' law (vocabulary grows sub-linearly with corpus size). Deviations reveal templated or synthetic text. The coverage curve answers a practical question: how many distinct words are needed to cover 90% or 95% of all tokens, which sets the vocabulary size for bag-of-words models.

In [ ]:
COUNT_ALL = Counter(t for doc in TOK_ALL for t in doc)
COUNT = Counter(t for doc in TOK for t in doc)
freq = np.array(sorted(COUNT_ALL.values(), reverse=True), dtype=float)
ranks = np.arange(1, len(freq) + 1)
fit_mask = (ranks >= 10) & (ranks <= min(10_000, len(ranks)))
zipf_slope = np.polyfit(np.log(ranks[fit_mask]), np.log(freq[fit_mask]), 1)[0] if fit_mask.sum() > 10 else np.nan

seen, growth_x, growth_y, total = set(), [], [], 0
step = max(1, len(TOK_ALL) // 200)
for i, doc in enumerate(TOK_ALL):
    seen.update(doc)
    total += len(doc)
    if i % step == 0:
        growth_x.append(total)
        growth_y.append(len(seen))
gx, gy = np.array(growth_x[1:], dtype=float), np.array(growth_y[1:], dtype=float)
gx, gy = gx[(gx > 0) & (gy > 0)], gy[(gx > 0) & (gy > 0)]
heaps_beta = np.polyfit(np.log(gx), np.log(gy), 1)[0] if len(gx) > 5 else np.nan

cov_all = np.cumsum(freq) / freq.sum()
freq_c = np.array(sorted(COUNT.values(), reverse=True), dtype=float)
cov_c = np.cumsum(freq_c) / freq_c.sum()
coverage = pd.DataFrame({'all_tokens': [int(np.searchsorted(cov_all, q) + 1) for q in (0.5, 0.8, 0.9, 0.95, 0.99)],
                         'content_tokens': [int(np.searchsorted(cov_c, q) + 1) for q in (0.5, 0.8, 0.9, 0.95, 0.99)]},
                        index=['50%', '80%', '90%', '95%', '99%'])
hapax = sum(1 for v in COUNT_ALL.values() if v == 1)
vocab_stats = pd.Series({'tokens': int(freq.sum()), 'vocabulary': len(COUNT_ALL), 'content vocabulary': len(COUNT),
                         'hapax legomena': hapax, 'hapax share of vocab': hapax / len(COUNT_ALL),
                         'zipf slope': zipf_slope, 'heaps beta': heaps_beta}, name='value')
save_table(vocab_stats.to_frame(), 'vocabulary_stats')
save_table(coverage, 'vocabulary_coverage')
display(vocab_stats.to_frame())
display(coverage.rename_axis('tokens covered'))

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].loglog(ranks, freq, color=PRIMARY, lw=2)
axes[0].loglog(ranks, freq[0] * ranks ** -1.0, color=ACCENT, ls='--', lw=1.2, label='ideal Zipf (slope -1)')
axes[0].set_xlabel('rank')
axes[0].set_ylabel('frequency')
axes[0].legend()
axes[0].set_title(f'Zipf plot (fitted slope {zipf_slope:.2f})', loc='left')
axes[1].plot(gx, gy, color=PALETTE[2], lw=2.2)
axes[1].set_xlabel('tokens seen')
axes[1].set_ylabel('distinct words')
axes[1].set_title(f'Heaps growth (beta {heaps_beta:.2f})', loc='left')
axes[2].semilogx(np.arange(1, len(cov_all) + 1), cov_all * 100, color=PRIMARY, lw=2, label='all tokens')
axes[2].semilogx(np.arange(1, len(cov_c) + 1), cov_c * 100, color=ACCENT, lw=2, label='content tokens')
for q in (90, 95):
    axes[2].axhline(q, color=MUTED, ls=':', lw=1)
axes[2].set_xlabel('top-N words (log)')
axes[2].set_ylabel('% of tokens covered')
axes[2].legend(loc='lower right')
axes[2].set_title(f'Coverage: {coverage.loc["95%", "content_tokens"]:,} content words cover 95%', loc='left')
suptitle(fig, 'Vocabulary statistics', f'{int(freq.sum()):,} tokens, {len(COUNT_ALL):,} distinct words, {hapax / len(COUNT_ALL):.0%} appear once')
save_fig(fig, 'vocabulary_statistics')

#### Insights

> *Fill in after running.* Does the corpus follow Zipf (slope near -1)? A large hapax share means many typos, names, or rare words, which favours subword tokenisers. How many words give 95% coverage, which is a sensible `max_features` for TF-IDF?

---

# Word Clouds <a name="8"></a>

---

Word clouds are the fastest way to communicate what a corpus talks about, which makes them ideal for the presentation. Word size is proportional to frequency after stopword removal. The overall cloud uses the full palette, the bigram cloud shows recurring phrases, and the per-label clouds reuse each label's colour. Section 10 adds clouds of *distinctive* words, which are usually more informative than raw frequency.

In [ ]:
def shades(color, n=6):
    base, dark = np.array(to_rgb(color)), np.array(to_rgb(DARK))
    return [to_hex(base * f + dark * (1 - f)) for f in np.linspace(0.55, 1.0, n)]

def color_func_from(colors):
    rng = random.Random(CFG.SEED)
    return lambda *args, **kwargs: rng.choice(colors)

def draw_cloud(ax, freqs, colors, title, max_words=None):
    freqs = {k: float(v) for k, v in freqs.items() if v > 0}
    if not freqs:
        ax.set_visible(False)
        return
    wc = WordCloud(width=1600, height=900, background_color='white', max_words=max_words or CFG.WORDCLOUD_WORDS,
                   prefer_horizontal=0.9, relative_scaling=0.5, min_font_size=8, random_state=CFG.SEED,
                   color_func=color_func_from(colors), collocations=False, margin=4)
    wc.generate_from_frequencies(freqs)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, loc='left', fontsize=13)
    ax.axis('off')

BIGRAMS = Counter(f'{a} {b}' for doc in TOK for a, b in zip(doc, doc[1:]) if a != b)

if not AVAILABLE['wordcloud']:
    note('wordcloud is not installed, so word clouds are skipped (pip install wordcloud).')
else:
    fig, axes = plt.subplots(1, 2, figsize=(22, 7))
    draw_cloud(axes[0], dict(COUNT.most_common(2000)), PALETTE, 'Most frequent words')
    draw_cloud(axes[1], dict(BIGRAMS.most_common(1000)), PALETTE[::-1], 'Most frequent bigrams', max_words=100)
    suptitle(fig, 'Corpus word clouds', f'{len(df):,} documents, stopwords removed (language: {CFG.LANGUAGE})')
    save_fig(fig, 'wordcloud_overall')

## Word Clouds per Label

Each panel shows the most frequent content words for one label in that label's colour. Words that dominate every panel are corpus-wide and are good candidates for `EXTRA_STOPWORDS`.

In [ ]:
LABEL_COUNTS = {}
if HAS_LABEL:
    lab_arr = df['label'].values
    for l in TOP_LABELS:
        LABEL_COUNTS[l] = Counter(t for doc, lb in zip(TOK, lab_arr) if lb == l for t in doc)
if HAS_LABEL and AVAILABLE['wordcloud']:
    fig, axes = make_grid(len(TOP_LABELS), ncols=2 if len(TOP_LABELS) <= 4 else 3, w=8.5, h=5.8)
    for ax, l in zip(axes, TOP_LABELS):
        n_docs = int((df['label'] == l).sum())
        draw_cloud(ax, dict(LABEL_COUNTS[l].most_common(1500)), shades(LABEL_COLOR[l]), f'{short(l, 30)}  ({n_docs:,} docs)', max_words=100)
    suptitle(fig, f'Word clouds by {LABEL_COL}', 'size = frequency within the label')
    save_fig(fig, 'wordcloud_by_label')

#### Insights

> *Fill in after running.* What themes dominate the corpus overall and per label? Which words appear everywhere and should be added to `EXTRA_STOPWORDS` before re-running?

---

# N-gram Analysis <a name="9"></a>

---

Bar charts give exact counts where word clouds give impressions. Unigrams show vocabulary, while bigrams and trigrams surface fixed phrases, negations, and templates (for example 'not working' in reviews or a repeated chorus line in lyrics). Counts are document frequencies, so one very repetitive document cannot dominate.

In [ ]:
def doc_freq(docs, n):
    c = Counter()
    for doc in docs:
        grams = set(zip(*[doc[i:] for i in range(n)])) if n > 1 else set(doc)
        c.update(' '.join(g) if n > 1 else g for g in grams)
    return c

NGRAMS = {n: doc_freq(TOK, n) for n in (1, 2, 3)}
fig, axes = plt.subplots(1, 3, figsize=(20, 0.33 * CFG.TOP_N + 2.4))
for ax, n, col in zip(axes, (1, 2, 3), (PRIMARY, ACCENT, PALETTE[2])):
    top = pd.Series(dict(NGRAMS[n].most_common(CFG.TOP_N))).sort_values()
    ax.barh([short(i, 30) for i in top.index], top.values / len(df) * 100, color=col)
    bar_labels(ax, '{:.1f}%', size=7)
    ax.set_xlabel('% of documents')
    ax.set_title(['Unigrams', 'Bigrams', 'Trigrams'][n - 1], loc='left')
    save_table(top[::-1].rename('documents').to_frame(), f'top_{n}grams')
suptitle(fig, 'Most common n-grams', 'document frequency after stopword removal')
save_fig(fig, 'ngrams_overall')

## Bigrams per Label

The most common phrases per label, shown as document share within that label, reveal how each group expresses itself.

In [ ]:
if HAS_LABEL:
    fig, axes = make_grid(len(TOP_LABELS), ncols=4, w=4.8, h=0.28 * 10 + 1.4)
    for ax, l in zip(axes, TOP_LABELS):
        docs = [d for d, lb in zip(TOK, df['label'].values) if lb == l]
        top = pd.Series(dict(doc_freq(docs, 2).most_common(10))).sort_values() / max(len(docs), 1) * 100
        ax.barh([short(i, 24) for i in top.index], top.values, color=LABEL_COLOR[l])
        ax.set_title(short(l, 28), loc='left', fontsize=10)
        ax.tick_params(axis='y', labelsize=8)
        ax.set_xlabel('% of docs in label', fontsize=8)
    suptitle(fig, f'Top bigrams by {LABEL_COL}', 'document share within each label')
    save_fig(fig, 'bigrams_by_label')

#### Insights

> *Fill in after running.* Which phrases recur, and do any look like boilerplate that should be removed? Which phrases are characteristic of specific labels?

---

# Distinctive Words per Label <a name="10"></a>

---

Raw frequency is dominated by words that are common everywhere. Distinctiveness is measured with the weighted log-odds ratio with an informative Dirichlet prior (Monroe, Colaresi and Quinn, 2008), which compares each label against all other documents and shrinks rare words toward the corpus background, so a word seen twice cannot top the list. Counts are document frequencies (how many documents use the word), so one document that repeats a word dozens of times cannot make it look characteristic. The z-score is the evidence that a word is over-used by the label.

$$\delta_w^{(i)} = \log\frac{y_w^{(i)} + \alpha_w}{n^{(i)} + \alpha_0 - y_w^{(i)} - \alpha_w} - \log\frac{y_w^{(j)} + \alpha_w}{n^{(j)} + \alpha_0 - y_w^{(j)} - \alpha_w}, \qquad z_w = \frac{\delta_w^{(i)}}{\sqrt{\frac{1}{y_w^{(i)} + \alpha_w} + \frac{1}{y_w^{(j)} + \alpha_w}}}$$

where $y_w^{(i)}$ is the number of documents in label $i$ containing word $w$, $j$ is the rest of the corpus, and the prior $\alpha_w$ is proportional to the background frequency of $w$.

In [ ]:
DISTINCT = {}
if not HAS_LABEL:
    note('No label column, so distinctive-word analysis is skipped.')
else:
    cv = CountVectorizer(analyzer=lambda d: d, min_df=5, max_features=30_000, binary=True)
    Xc = cv.fit_transform(TOK)
    vocab = cv.get_feature_names_out()
    total = np.asarray(Xc.sum(axis=0)).ravel().astype(float)
    a0 = 10_000.0
    alpha = total / total.sum() * a0
    rows = []
    for l in TOP_LABELS:
        m = (df['label'] == l).values
        yi = np.asarray(Xc[m].sum(axis=0)).ravel()
        yj = total - yi
        ni, nj = yi.sum(), yj.sum()
        delta = np.log((yi + alpha) / (ni + a0 - yi - alpha)) - np.log((yj + alpha) / (nj + a0 - yj - alpha))
        z = delta / np.sqrt(1 / (yi + alpha) + 1 / (yj + alpha))
        DISTINCT[l] = pd.Series(z, index=vocab).sort_values(ascending=False)
        for w in DISTINCT[l].index[:30]:
            rows.append({'label': l, 'word': w, 'z': DISTINCT[l][w], 'docs_in_label': int(yi[cv.vocabulary_[w]])})
    distinct_table = pd.DataFrame(rows)
    save_table(distinct_table, 'distinctive_words', index=False)

    fig, axes = make_grid(len(TOP_LABELS), ncols=4, w=4.6, h=0.27 * 12 + 1.4)
    for ax, l in zip(axes, TOP_LABELS):
        top = DISTINCT[l].head(12)[::-1]
        ax.barh(top.index, top.values, color=LABEL_COLOR[l])
        ax.set_title(short(l, 28), loc='left', fontsize=10)
        ax.tick_params(axis='y', labelsize=8)
        ax.set_xlabel('log-odds z-score', fontsize=8)
    suptitle(fig, f'Most distinctive words by {LABEL_COL}', 'weighted log-odds with informative Dirichlet prior, each label vs the rest')
    save_fig(fig, 'distinctive_words')

## Distinctive Word Clouds

The same z-scores drive a second set of word clouds where size reflects how characteristic a word is rather than how frequent. These are usually the clearest visual for a slide about 'the lyrical profile of each genre' or 'what each customer segment complains about'.

In [ ]:
if DISTINCT and AVAILABLE['wordcloud']:
    fig, axes = make_grid(len(TOP_LABELS), ncols=2 if len(TOP_LABELS) <= 4 else 3, w=8.5, h=5.8)
    for ax, l in zip(axes, TOP_LABELS):
        z = DISTINCT[l]
        draw_cloud(ax, z[z > 1.96].head(150).to_dict(), shades(LABEL_COLOR[l]), f'{short(l, 30)}: distinctive words', max_words=100)
    suptitle(fig, f'Distinctive word clouds by {LABEL_COL}', 'size = log-odds z-score (only z > 1.96, i.e. significant over-use)')
    save_fig(fig, 'wordcloud_distinctive')

## Two-Label Comparison

For the two largest labels, each word is placed by its frequency in label A (x) and label B (y). Words far from the diagonal belong to one side. The most distinctive words on each side are annotated, in the spirit of a Scattertext plot.

In [ ]:
if HAS_LABEL and len(TOP_LABELS) >= 2:
    A, B = TOP_LABELS[0], TOP_LABELS[1]
    ca, cb = LABEL_COUNTS[A], LABEL_COUNTS[B]
    words = [w for w, _ in (ca + cb).most_common(3000)]
    na, nb = sum(ca.values()), sum(cb.values())
    fa = np.array([(ca[w] + 1) / na * 1e4 for w in words])
    fb = np.array([(cb[w] + 1) / nb * 1e4 for w in words])
    lr = np.log2(fa / fb)
    fig, ax = plt.subplots(figsize=(11, 10))
    sc = ax.scatter(fa, fb, c=lr, cmap=LinearSegmentedColormap.from_list('ab', [LABEL_COLOR[B], '#E2E8F0', LABEL_COLOR[A]]),
                    vmin=-3, vmax=3, s=12, alpha=0.75)
    ax.set_xscale('log')
    ax.set_yscale('log')
    lims = [min(fa.min(), fb.min()), max(fa.max(), fb.max())]
    ax.plot(lims, lims, color=MUTED, ls='--', lw=1)
    weight = np.log1p(fa + fb)
    for idx in list(np.argsort(-lr * weight)[:12]) + list(np.argsort(lr * weight)[:12]):
        ax.annotate(words[idx], (fa[idx], fb[idx]), fontsize=8, color=DARK, xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel(f'frequency in {A} (per 10k tokens)')
    ax.set_ylabel(f'frequency in {B} (per 10k tokens)')
    fig.colorbar(sc, ax=ax, shrink=0.6, label=f'log2 ratio ({A} / {B})')
    suptitle(fig, f'{A} vs {B}', 'top 3,000 words; annotated = most one-sided frequent words')
    save_fig(fig, 'two_label_comparison')

#### Insights

> *Fill in after running.* Which words define each label? These distinctive vocabularies are the core of a 'profile per category' narrative and also predict which labels a bag-of-words model will separate easily (strong, unique vocabulary) or confuse (overlapping vocabulary).

---

# Sentiment Analysis <a name="11"></a>

---

VADER is a rule-based sentiment lexicon tuned for short, informal English text. It needs no training and returns a compound score from -1 (most negative) to +1 (most positive), with +/-0.05 as the conventional neutral band. It is applied to a sample of `SENTIMENT_SAMPLE` documents. For Indonesian text a lexicon is unreliable, and a fine-tuned IndoBERT sentiment model in the NLP modelling notebook is the right tool.

In [ ]:
SENT = None
if not AVAILABLE['vader']:
    note('vaderSentiment is not installed, so sentiment analysis is skipped.')
elif CFG.LANGUAGE == 'id':
    note('LANGUAGE is Indonesian, and VADER is English-only, so lexicon sentiment is skipped.')
else:
    vader = SentimentIntensityAnalyzer()
    s_idx = df.sample(min(CFG.SENTIMENT_SAMPLE, len(df)), random_state=CFG.SEED).index
    def strip_patterns(t):
        for rx in REMOVE_RES:
            t = rx.sub(' ', t)
        return ' '.join(t.split())
    texts = [strip_patterns(t) for t in df.loc[s_idx, 'text']]
    SENT = pd.DataFrame([vader.polarity_scores(t[:3000]) for t in texts], index=s_idx)
    SENT['polarity'] = pd.cut(SENT['compound'], [-1.01, -0.05, 0.05, 1.01], labels=['negative', 'neutral', 'positive'])
    if HAS_LABEL:
        SENT['label'] = df.loc[s_idx, 'label']
    save_table(SENT, 'vader_scores')
    share = SENT['polarity'].value_counts(normalize=True).reindex(['negative', 'neutral', 'positive']) * 100
    print('Polarity share (%):', share.round(1).to_dict())

    pol_color = {'negative': ACCENT, 'neutral': '#C9D1DB', 'positive': PALETTE[2]}
    ncols = 2 if HAS_LABEL else 1
    fig, axes = plt.subplots(1, ncols, figsize=(8.5 * ncols, 5.2), squeeze=False)
    ax = axes[0, 0]
    sns.histplot(SENT['compound'], bins=40, color=PRIMARY, edgecolor='white', ax=ax)
    ax.axvspan(-0.05, 0.05, color='#E2E8F0', alpha=0.8)
    ax.set_title(f'Compound score: {share["positive"]:.0f}% pos, {share["neutral"]:.0f}% neu, {share["negative"]:.0f}% neg', loc='left')
    if HAS_LABEL:
        d = SENT[SENT['label'].isin(TOP_LABELS)]
        ct = pd.crosstab(d['label'], d['polarity'], normalize='index').reindex(columns=['negative', 'neutral', 'positive']).fillna(0)
        ct = ct.sort_values('positive')
        left = np.zeros(len(ct))
        for pcol in ct.columns:
            axes[0, 1].barh([short(i, 22) for i in ct.index], ct[pcol], left=left, color=pol_color[pcol], label=pcol, edgecolor='white')
            left += ct[pcol].values
        axes[0, 1].set_xlim(0, 1)
        axes[0, 1].legend(loc='lower right', fontsize=8)
        axes[0, 1].set_title('Polarity mix per label (sorted by positive share)', loc='left')
        save_table(ct, 'polarity_by_label')
    suptitle(fig, 'Lexicon sentiment (VADER)', f'{len(SENT):,} sampled documents, first 3,000 characters each')
    save_fig(fig, 'sentiment')
    if HAS_LABEL and LABEL_KIND == 'binned numeric':
        rho, p = stats.spearmanr(SENT['compound'], df.loc[s_idx, LABEL_COL], nan_policy='omit')
        print(f'Spearman rho between compound sentiment and {LABEL_COL}: {rho:+.3f} (p = {p:.2g})')

#### Insights

> *Fill in after running.* What is the overall tone of the corpus, and which labels are the most positive or negative? Remember that lexicons miss sarcasm and domain meaning (for example 'killer' as praise), so treat these as indicative and validate with examples.

---

# Topic Modeling <a name="12"></a>

---

Non-negative Matrix Factorisation on TF-IDF decomposes the corpus into `TOPIC_K` additive topics, each a weighted list of words, and gives every document a mixture of topics. NMF is fast, deterministic with a fixed seed, and produces more coherent topics than LDA on short or medium texts. Words that appear in more than half of the documents are excluded so topics are not dominated by generic vocabulary.

In [ ]:
t_idx = df.sample(min(20_000, len(df)), random_state=CFG.SEED).index
tfv = TfidfVectorizer(analyzer=lambda d: d, min_df=5, max_df=0.5, max_features=20_000, sublinear_tf=True)
Xt = tfv.fit_transform([TOK[i] for i in t_idx])
nmf = NMF(n_components=CFG.TOPIC_K, init='nndsvda', random_state=CFG.SEED, max_iter=400)
W = nmf.fit_transform(Xt)
H = nmf.components_
terms = tfv.get_feature_names_out()
TOPIC_WORDS = [[terms[j] for j in np.argsort(-H[k])[:10]] for k in range(CFG.TOPIC_K)]
TOPIC_NAMES = [f'T{k + 1}: ' + ', '.join(w[:3]) for k, w in enumerate(TOPIC_WORDS)]
Wn = W / np.clip(W.sum(axis=1, keepdims=True), 1e-12, None)
topics = pd.DataFrame({'topic': TOPIC_NAMES, 'top_words': [', '.join(w) for w in TOPIC_WORDS],
                       'prevalence_%': Wn.mean(axis=0) * 100}).set_index('topic')
save_table(topics, 'nmf_topics')
display(topics.round(2))

fig, axes = make_grid(CFG.TOPIC_K, ncols=4, w=4.3, h=3.5)
for k, ax in enumerate(axes):
    idx = np.argsort(-H[k])[:10][::-1]
    ax.barh(terms[idx], H[k][idx], color=PALETTE[k % 10])
    ax.set_title(f'Topic {k + 1} ({topics["prevalence_%"].iloc[k]:.1f}%)', loc='left', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_xticks([])
suptitle(fig, f'NMF topics (k = {CFG.TOPIC_K})', f'top 10 words per topic; % = mean topic share across {len(t_idx):,} documents')
save_fig(fig, 'nmf_topics')

## Topic Mix per Label

The heatmap shows the average topic share for each label, normalised so each row sums to 1. Bright cells mark the themes a label over-represents.

In [ ]:
if HAS_LABEL:
    labs = df.loc[t_idx, 'label'].values
    rows = {l: Wn[labs == l].mean(axis=0) for l in TOP_LABELS if (labs == l).any()}
    mix = pd.DataFrame(rows, index=TOPIC_NAMES).T
    lift = mix / Wn.mean(axis=0)
    save_table(mix, 'topic_mix_by_label')
    fig, ax = plt.subplots(figsize=(max(10, 1.4 * CFG.TOPIC_K + 3), 0.5 * len(mix) + 2.4))
    sns.heatmap(lift, cmap=CMAP_HEAT, annot=mix * 100, fmt='.0f', linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'lift vs corpus average', 'shrink': 0.7}, ax=ax,
                xticklabels=[short(t, 24) for t in TOPIC_NAMES], yticklabels=[short(l, 22) for l in mix.index])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('')
    suptitle(fig, f'Topic mix by {LABEL_COL}', 'cell text = % topic share, colour = lift over the corpus average')
    save_fig(fig, 'topic_mix_by_label')

#### Insights

> *Fill in after running.* Name each topic in plain language. Which labels are dominated by which themes? If topics look incoherent, try another `TOPIC_K` or add generic words to `EXTRA_STOPWORDS`.

---

# Semantic Map <a name="13"></a>

---

A 2-D map shows whether labels occupy distinct regions of meaning. Documents are embedded with TF-IDF, compressed to 50 dimensions with truncated SVD (latent semantic analysis), and projected with t-SNE. Well-separated clusters predict an easy classification task, while heavy overlap predicts confusion between labels.

In [ ]:
m_idx = df[df['n_content'] > 0].sample(min(CFG.TSNE_SAMPLE, int((df['n_content'] > 0).sum())), random_state=CFG.SEED).index
Xm = TfidfVectorizer(analyzer=lambda d: d, min_df=3, max_df=0.5, sublinear_tf=True).fit_transform([TOK[i] for i in m_idx])
Z = TruncatedSVD(n_components=min(50, Xm.shape[1] - 1), random_state=CFG.SEED).fit_transform(Xm)
Y = TSNE(n_components=2, perplexity=30, init='pca', random_state=CFG.SEED).fit_transform(Z)

fig, ax = plt.subplots(figsize=(12, 9))
if HAS_LABEL:
    labs = df.loc[m_idx, 'label_top'].values
    other = labs == '(other)'
    ax.scatter(Y[other, 0], Y[other, 1], s=6, color='#D5DBE3', alpha=0.5, label='(other)')
    for l in TOP_LABELS:
        m = labs == l
        ax.scatter(Y[m, 0], Y[m, 1], s=9, color=LABEL_COLOR[l], alpha=0.75, label=short(l, 20))
    ax.legend(markerscale=2.5, fontsize=8, loc='upper left', bbox_to_anchor=(1, 1))
else:
    dom = nmf.transform(tfv.transform([TOK[i] for i in m_idx])).argmax(axis=1)
    for k in range(CFG.TOPIC_K):
        m = dom == k
        ax.scatter(Y[m, 0], Y[m, 1], s=9, color=PALETTE[k % 10], alpha=0.75, label=short(TOPIC_NAMES[k], 26))
    ax.legend(markerscale=2.5, fontsize=8, loc='upper left', bbox_to_anchor=(1, 1))
ax.set_xticks([])
ax.set_yticks([])
colour_by = LABEL_COL if HAS_LABEL else 'dominant NMF topic'
suptitle(fig, 'Semantic map (TF-IDF -> SVD -> t-SNE)', f'{len(m_idx):,} sampled documents coloured by {colour_by}')
save_fig(fig, 'semantic_map')

#### Insights

> *Fill in after running.* Do labels form separate clusters or one mixed cloud? Overlap at the bag-of-words level suggests that contextual embeddings (transformers) will be needed to separate them.

---

# Duplicates and Label Noise <a name="14"></a>

---

Duplicate texts inflate validation scores when copies fall into both training and validation folds, and identical texts with different labels cap the achievable accuracy. Exact duplicates are found after normalisation. Near duplicates (reposts, small edits, remixes) are found with cosine similarity of TF-IDF word and bigram vectors on a sample.

In [ ]:
norm = df['clean'].str.split().str.join(' ')
valid = norm != ''
grp = df[valid].groupby(norm[valid])
sizes = grp.size()
dup_groups = sizes[sizes > 1]
print(f'Exact duplicate groups: {len(dup_groups):,} covering {dup_groups.sum():,} documents ({dup_groups.sum() / len(df):.1%})')
CONFLICTS = 0
if HAS_LABEL and len(dup_groups):
    n_lab = grp['label'].nunique()
    conflict_keys = n_lab[n_lab > 1].index
    CONFLICTS = len(conflict_keys)
    print(f'Duplicate groups with conflicting labels: {CONFLICTS:,} ({CONFLICTS / max(len(dup_groups), 1):.1%} of duplicate groups)')
    ex = [{'text': short(k, 90), 'copies': int(sizes[k]), 'labels': ', '.join(sorted(df.loc[grp.groups[k], 'label'].unique()))[:80]}
          for k in list(conflict_keys[:8])]
    if ex:
        display(pd.DataFrame(ex))

n_idx = df[valid].sample(min(CFG.NEAR_DUP_SAMPLE, int(valid.sum())), random_state=CFG.SEED).index
Xn = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(norm[n_idx])
nn = NearestNeighbors(n_neighbors=2, metric='cosine').fit(Xn)
dist, nbr = nn.kneighbors(Xn)
sim = 1 - dist[:, 1]
exact = norm[n_idx].values == norm[n_idx].values[nbr[:, 1]]
near = (sim >= CFG.NEAR_DUP_SIM) & ~exact
NEAR_SHARE = near.mean()
print(f'Near duplicates (cosine >= {CFG.NEAR_DUP_SIM}, not exact) in sample: {near.sum():,} ({NEAR_SHARE:.1%})')

fig, ax = plt.subplots(figsize=(11, 4.4))
sns.histplot(sim, bins=50, color=PRIMARY, edgecolor='white', ax=ax)
ax.axvline(CFG.NEAR_DUP_SIM, color=ACCENT, ls='--', lw=1.5, label=f'near-duplicate threshold {CFG.NEAR_DUP_SIM}')
ax.set_xlabel('cosine similarity to nearest other document')
ax.legend()
suptitle(fig, 'Nearest-neighbour similarity', f'{len(n_idx):,} sampled documents; mass near 1.0 = duplicated content')
save_fig(fig, 'near_duplicates')

pairs = [{'similarity': round(float(sim[i]), 3), 'text_a': short(norm[n_idx[i]], 70), 'text_b': short(norm[n_idx[nbr[i, 1]]], 70),
          **({'label_a': df.loc[n_idx[i], 'label'], 'label_b': df.loc[n_idx[nbr[i, 1]], 'label']} if HAS_LABEL else {})}
         for i in np.where(near)[0][:8]]
if pairs:
    display(pd.DataFrame(pairs))

#### Insights

> *Fill in after running.* How much of the corpus is duplicated, and do duplicates carry conflicting labels? Duplicates must be removed or kept together with a group-aware split. Conflicting labels set an upper bound on accuracy and should be reported as label noise.

---

# Segment and Temporal Analysis <a name="15"></a>

---

When a segment column (`GROUP_COL`) or a time column (`TIME_COL`) is available, text properties are tracked across them: document volume, length, the usage of the most frequent words, and the label mix. This reveals drift, for example reviews getting longer after an app update or vocabulary shifting over the years.

In [ ]:
HAS_TIME = bool(CFG.TIME_COL) and CFG.TIME_COL in df.columns
if HAS_TIME:
    tc = df[CFG.TIME_COL]
    if pd.api.types.is_numeric_dtype(tc):
        period = tc.where(tc.between(*CFG.TIME_RANGE)).round()
        if period.nunique() > 30:
            period = (period // 5 * 5)
            p_label = f'{CFG.TIME_COL} (5-year bins)'
        else:
            p_label = CFG.TIME_COL
    else:
        dt = pd.to_datetime(tc, errors='coerce')
        span = (dt.max() - dt.min()).days
        freq = 'D' if span <= 90 else 'W' if span <= 730 else 'M' if span <= 3650 else 'Y'
        period = dt.dt.to_period(freq).dt.to_timestamp()
        p_label = f'{CFG.TIME_COL} ({freq})'
    df['period'] = period
    d = df.dropna(subset=['period'])
    agg = d.groupby('period').agg(docs=('text', 'size'), median_words=('n_tokens', 'median'),
                                  q25=('n_tokens', lambda s: s.quantile(0.25)), q75=('n_tokens', lambda s: s.quantile(0.75)))
    agg = agg[agg['docs'] >= 10]
    top_words = [w for w, _ in COUNT.most_common(6)]
    rate = {}
    for p_, g in d.groupby('period'):
        if len(g) < 10:
            continue
        c = Counter(t for i in g.index for t in TOK[i])
        n_tok = max(sum(c.values()), 1)
        rate[p_] = {w: c[w] / n_tok * 1000 for w in top_words}
    rate = pd.DataFrame(rate).T.sort_index()

    ncols = 3 if HAS_LABEL else 2
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 5))
    width = 3.5 if p_label.endswith('bins)') else 0.8
    if pd.api.types.is_datetime64_any_dtype(agg.index):
        width = 20
    axes[0].bar(agg.index, agg['docs'], color=SOFT, width=width)
    ax2 = axes[0].twinx()
    ax2.fill_between(agg.index, agg['q25'], agg['q75'], color=ACCENT, alpha=0.15)
    ax2.plot(agg.index, agg['median_words'], color=ACCENT, marker='o', ms=3, lw=2)
    ax2.set_ylabel('median words (band = IQR)', color=ACCENT)
    ax2.grid(False)
    axes[0].set_ylabel('documents')
    axes[0].set_title('Volume and length over time', loc='left')
    for i, w in enumerate(top_words):
        axes[1].plot(rate.index, rate[w], marker='o', ms=3, lw=1.8, color=PALETTE[i % 10], label=w)
    axes[1].set_ylabel('uses per 1,000 content tokens')
    axes[1].legend(fontsize=8)
    axes[1].set_title('Top words over time', loc='left')
    if HAS_LABEL:
        ct = pd.crosstab(d['period'], d['label_top'], normalize='index')
        ct = ct.loc[ct.index.isin(agg.index)]
        cols = [l for l in TOP_LABELS if l in ct.columns] + (['(other)'] if '(other)' in ct.columns else [])
        axes[2].stackplot(ct.index, ct[cols].T.values, colors=[LABEL_COLOR.get(c, '#D5DBE3') for c in cols],
                          labels=[short(c, 18) for c in cols], alpha=0.9)
        axes[2].set_ylim(0, 1)
        axes[2].legend(fontsize=7, loc='upper left', bbox_to_anchor=(1, 1))
        axes[2].set_title('Label mix over time', loc='left')
    for ax in axes:
        ax.set_xlabel(p_label)
    suptitle(fig, f'Text over {CFG.TIME_COL}', 'periods with fewer than 10 documents are hidden')
    save_fig(fig, 'temporal_text')
    save_table(agg.join(rate), 'text_over_time')
else:
    note('TIME_COL is not set or not found, so temporal analysis is skipped.')

if CFG.GROUP_COL and CFG.GROUP_COL in df.columns:
    g = df.groupby(CFG.GROUP_COL)
    seg = g.agg(docs=('text', 'size'), median_words=('n_tokens', 'median'), ttr=('ttr', 'median')).sort_values('docs', ascending=False).head(20)
    save_table(seg, 'text_by_group')
    display(seg)
    fig, axes = plt.subplots(1, 2, figsize=(16, 0.36 * len(seg) + 2))
    axes[0].barh([short(i, 22) for i in seg.index[::-1]], seg['docs'][::-1], color=PRIMARY)
    axes[0].set_title('Documents per group', loc='left')
    axes[1].barh([short(i, 22) for i in seg.index[::-1]], seg['median_words'][::-1], color=ACCENT)
    axes[1].set_title('Median words per group', loc='left')
    suptitle(fig, f'Text by {CFG.GROUP_COL}', 'top 20 groups by size')
    save_fig(fig, 'text_by_group')

#### Insights

> *Fill in after running.* Does the language or the label mix drift over time or across segments? If so, a random split overstates performance on future data and a time-based or group-based split is more honest.

---

# EDA Summary and Modelling Recommendations <a name="16"></a>

---

The key findings are compiled automatically and translated into concrete modelling settings: the transformer `max_length` (p95 of words times about 1.3 subword tokens per word), the TF-IDF vocabulary size (words for 95% coverage), the cleaning steps implied by the noise audit, and the handling of imbalance and duplicates. The summary is saved as `eda-output/eda_summary.md`.

In [ ]:
p95 = df['n_tokens'].quantile(0.95)
max_len = int(min(512, 2 ** np.ceil(np.log2(max(p95 * 1.3, 16)))))
lines = [f'# NLP EDA Summary: {DATA_PATH.name} (`{TEXT_COL}`)', '', '## Corpus',
         f'- {len(df):,} documents analysed (file has {N_TOTAL:,}); median {df.n_tokens.median():.0f} words, p95 {p95:.0f}, max {df.n_tokens.max():,}',
         f'- Empty documents: {int(overview.loc["empty or whitespace", "value"]):,}; exact duplicates: {int(overview.loc["duplicates after normalisation", "value"]):,} '
         f'({overview.loc["duplicates after normalisation", "share_%"]:.1f}%); near duplicates in sample: {NEAR_SHARE:.1%}',
         f'- Language guess: ' + ', '.join(f'{k} {v / len(df):.0%}' for k, v in lang.items()),
         f'- Vocabulary: {len(COUNT_ALL):,} words, {hapax / len(COUNT_ALL):.0%} hapax, Zipf slope {zipf_slope:.2f}', '']
if HAS_LABEL:
    vc = df['label'].value_counts()
    lines += ['## Labels', f'- `{LABEL_COL}` ({LABEL_KIND}): {len(vc)} labels, imbalance {vc.max() / vc.min():.1f}x',
              f'- Duplicate groups with conflicting labels: {CONFLICTS:,}']
    for l in TOP_LABELS[:8]:
        if l in DISTINCT:
            lines.append(f'- **{l}** distinctive words: ' + ', '.join(DISTINCT[l].index[:8]))
    lines.append('')
if SENT is not None:
    sh = SENT['polarity'].value_counts(normalize=True)
    lines += ['## Sentiment (VADER)', f'- positive {sh.get("positive", 0):.0%}, neutral {sh.get("neutral", 0):.0%}, negative {sh.get("negative", 0):.0%}', '']
lines += ['## Topics (NMF)'] + [f'- {t}: {w}' for t, w in topics['top_words'].items()] + ['']

recs = [f'Transformer `max_length` = {max_len} tokens (p95 = {p95:.0f} words x 1.3 subwords/word, capped at 512)'
        + ('; consider head+tail truncation or chunking for long documents' if p95 * 1.3 > 512 else ''),
        f'TF-IDF `max_features` around {int(coverage.loc["95%", "content_tokens"]):,} (95% content-token coverage)']
heavy = noise[noise['docs_%'] > 1].index.tolist()
if heavy:
    recs.append('Handle frequent noise patterns: ' + ', '.join(heavy))
if HAS_LABEL:
    vc = df['label'].value_counts()
    if vc.max() / vc.min() > 3:
        recs.append(f'Imbalance {vc.max() / vc.min():.1f}x: use stratified folds, class weights, and macro F1')
if overview.loc['duplicates after normalisation', 'share_%'] > 1 or NEAR_SHARE > 0.01:
    recs.append('Deduplicate, or split with groups of identical / near-identical texts, to avoid leakage')
if CONFLICTS:
    recs.append(f'{CONFLICTS:,} duplicate texts carry different labels: treat as label noise (resolve, drop, or use multi-label)')
other_lang = 1 - (lang.get('English', 0) if CFG.LANGUAGE == 'en' else lang.get('Indonesian', 0) if CFG.LANGUAGE == 'id' else lang.get('English', 0) + lang.get('Indonesian', 0)) / len(df)
if other_lang > 0.1:
    recs.append(f'{other_lang:.0%} of documents are outside the configured language: prefer a multilingual encoder (e.g. XLM-R, mE5)')
lines += ['## Modelling recommendations'] + [f'{i + 1}. {r}' for i, r in enumerate(recs)]
lines += ['', '## Exported files', f'- Figures ({len(SAVED_FIGS)}): `{CFG.FIG_DIR}`', f'- Tables: `{CFG.TABLE_DIR}`']
summary_md = '\n'.join(lines)
(CFG.OUTPUT_DIR / 'eda_summary.md').write_text(summary_md, encoding='utf-8')
display(Markdown(summary_md))

## Decisions File for the Pipeline

The key decisions are also written as machine-readable JSON to `eda-output/eda_decisions.json`. The NLP pipeline notebook reads this file and uses it for every setting left on `'auto'` (language and model family, `max_length`, TF-IDF size, cleaning patterns, class weighting, and duplicate handling), so the modelling choices are traceable to EDA evidence.

In [ ]:
lang_share = (lang / len(df)).to_dict()
en_s, id_s = lang_share.get('English', 0), lang_share.get('Indonesian', 0)
dominant = 'id' if id_s >= 0.6 else 'en' if en_s >= 0.6 else 'multi'
family = {'id': 'indobert', 'en': 'deberta-v3', 'multi': 'xlm-roberta'}[dominant]
imb = float(df['label'].value_counts().max() / df['label'].value_counts().min()) if HAS_LABEL else None
decisions = {
    'source': str(DATA_PATH), 'text_col': TEXT_COL, 'label_col': LABEL_COL, 'label_kind': LABEL_KIND,
    'language': dominant, 'language_share': {k: round(float(v), 4) for k, v in lang_share.items()},
    'recommended_model': family, 'max_length': max_len,
    'words_p50': float(df['n_tokens'].median()), 'words_p95': float(p95),
    'tfidf_max_features': int(coverage.loc['95%', 'content_tokens']),
    'remove_patterns': list(CFG.REMOVE_PATTERNS), 'extra_stopwords': list(CFG.EXTRA_STOPWORDS),
    'noise_patterns': heavy, 'imbalance_ratio': imb, 'class_weights': bool(imb and imb > 3),
    'duplicate_share': float(overview.loc['duplicates after normalisation', 'share_%'] / 100),
    'near_duplicate_share': float(NEAR_SHARE), 'conflicting_label_groups': int(CONFLICTS),
    'group_identical_texts': bool(overview.loc['duplicates after normalisation', 'share_%'] > 1 or NEAR_SHARE > 0.01),
}
(CFG.OUTPUT_DIR / 'eda_decisions.json').write_text(json.dumps(decisions, indent=2), encoding='utf-8')
print(json.dumps(decisions, indent=2))

The final cell lists every exported figure and table.

In [ ]:
print(f'Figures in {CFG.FIG_DIR.resolve()}:')
for f in SAVED_FIGS:
    print('  ', f)
print(f'\nTables in {CFG.TABLE_DIR.resolve()}:')
for f in sorted(CFG.TABLE_DIR.glob('*.csv')):
    print('  ', f.name)

#### Insights

> *Fill in after running.* Summarise the three to five findings that shape preprocessing, the model choice, validation, and the business narrative, each linked to its figure.